# NB05d — Supervised Neural Networks with PyTorch

**Role:** Compact-extension  
**Manual section:** 2.3.1.d — Neural networks and deep learning awareness  
**Prerequisites:** NB05

---

## Purpose of this notebook

This compact-extension notebook demystifies neural networks with a minimal MLP (multi-layer perceptron) example using PyTorch. The goal is not to optimise the network but to show the workflow: define architecture, train with a loss function, and compare against a logistic regression baseline.

For a broader conceptual overview of neural networks in finance (without PyTorch code), see **NB06c**.

**Dataset:** `dataset_F_deep_learning_awareness.csv`

## Section 0 — Why include a neural-network example

Neural networks are important in modern AI, but in many tabular finance problems they are not automatically better than strong tree-based methods. This notebook is intentionally compact: the goal is to demystify the workflow, not to build a deep-learning engineering course.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from pathlib import Path
_candidates = [Path('data/processed'), Path('../data/processed')]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Cannot locate data/processed/. Launch from project root or notebooks/.')

import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression


In [ ]:
df = pd.read_csv(DATA_DIR / 'dataset_F_deep_learning_awareness.csv').copy()
X = pd.get_dummies(df.drop(columns=['customer_id', 'attrition_flag']), drop_first=True)
y = df['attrition_flag'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype('float32')
X_test_sc = scaler.transform(X_test).astype('float32')
X_train_sc.shape

## Section 1 — Logistic baseline on the same split

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_sc, y_train)
prob_lr = logreg.predict_proba(X_test_sc)[:, 1]
pred_lr = logreg.predict(X_test_sc)

def metric_row(name, y_true, y_pred, y_prob):
    return {'model': name, 'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'auc': roc_auc_score(y_true, y_prob)}

res_lr = metric_row('Logistic baseline', y_test, pred_lr, prob_lr)
res_lr

## Section 2 — Small PyTorch MLP

In [ ]:
torch.manual_seed(42)
X_train_t = torch.tensor(X_train_sc)
y_train_t = torch.tensor(y_train.reshape(-1, 1)).float()
X_test_t = torch.tensor(X_test_sc)

model = nn.Sequential(nn.Linear(X_train_sc.shape[1], 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1))
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_history = []
for epoch in range(12):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))
loss_history[:3], loss_history[-3:]

### A note on the number of epochs

This notebook intentionally uses a small number of training epochs. The goal here is to demonstrate the **PyTorch training workflow** — not to optimise the neural network's performance. In practice, you would typically train for more epochs, monitor a validation loss to detect overfitting, and potentially use early stopping.

The comparison with logistic regression below should therefore be read as a **workflow comparison** rather than a definitive performance verdict.

In [ ]:
plt.plot(loss_history)
plt.title('Training loss — small PyTorch MLP')
plt.xlabel('Epoch')
plt.ylabel('BCE loss')
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    logits_test = model(X_test_t).numpy().ravel()
prob_nn = 1 / (1 + np.exp(-logits_test))
pred_nn = (prob_nn >= 0.5).astype(int)
res_nn = metric_row('PyTorch MLP', y_test, pred_nn, prob_nn)
res_nn

### What to expect from the comparison

We now compare the neural network with a logistic regression baseline on the same test set. Since we used few training epochs and a minimal architecture, the neural network may not outperform logistic regression — and that is precisely the point.

For small, structured tabular datasets like those common in finance, simpler models often perform comparably to neural networks. The value of this notebook is understanding the **workflow**, not proving that neural networks are always better.

## Section 3 — Compare results

In [ ]:
comparison = pd.DataFrame([res_lr, res_nn]).round(3)
comparison

In [ ]:
comparison.set_index('model')[['auc', 'recall', 'f1']].plot(kind='bar')
plt.ylim(0, 1)
plt.title('Logistic baseline vs. small PyTorch MLP')
plt.xticks(rotation=10)
plt.show()

## What you have learned

- A PyTorch MLP follows a clear workflow: define layers, choose a loss function, run a training loop, evaluate on held-out data.
- Neural networks are not automatically better than simpler models for small tabular datasets.
- The training workflow (forward pass → loss → backward pass → weight update) is the conceptual foundation for all deep learning, regardless of architecture.
- For a broader conceptual overview of when deep learning helps and when it does not, see **NB06c**.

---

**Project chain:** NB05 (baseline) → NB05b (trees) → **NB05d (neural network demo)**  
**Current position:** NB05d

*This is a compact-extension notebook supporting manual section 2.3.1.d.*